# 🩺 Amar Doctor V1 — AI Video & Neural Voice Sandbox (Colab GPU)
### Free-tier AI telemedicine backend running on Google Colab
- 🎙️ **Edge-TTS Bengali Neural Voice Synthesis** (`bn-BD-NabanitaNeural` / `bn-BD-PradeepNeural`)
- 🧠 **faster-whisper `large-v3` Bengali speech recognition (STT)** — self-hosted, no browser/cloud Web Speech API
- 🤖 **Groq (GPT-OSS-120B) medical triage engine**
- 📹 **MuseTalk GPU lip-sync avatar (optional)** — real inpainted video, not a looping clip
- 🌐 **Free Cloudflare Tunnel** for a public HTTPS URL your local Next.js app can call

**Two independent pieces, and you only need the first one:**

| | Needs | Cell(s) |
|---|---|---|
| 🎙️ Voice calls | just a T4 (or better) GPU + a Groq API key | 1, 2, 3, 7 |
| 📹 Video calls with real GPU lip-sync | the above, **plus** the MuseTalk environment (~7GB, one-time) **and** a doctor portrait/clip | all of them |

Skip the MuseTalk cells (or leave `ENABLE_MUSETALK = "0"` in cell 3) and everything still works —
video calls just show an audio-reactive avatar instead of GPU-rendered lip-sync, exactly like running
the backend locally without the sidecar. See [`MUSETALK_SETUP.md`](../MUSETALK_SETUP.md) for the same
recipe on Windows and for what these numbers mean on your own GPU.

**Colab's free tier disconnects often** (idle timeout, ~12h session cap). Cell 4 mounts Google Drive
so the MuseTalk install (venv + 7GB of weights + the prepared avatar) survives a restart instead of
redoing itself — strongly recommended if you're using video calls.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Clone your repository
!git clone https://github.com/CHANGE-ME/amar-doctor.git /content/amar-doctor  # <-- set your repo URL
%cd /content/amar-doctor

## 3. Configure this session
`GROQ_API_KEY` is required for every mode — without it every reply is a canned fallback that ignores
what the patient said. `ENABLE_MUSETALK` controls whether `colab_runner.py` (cell 7) starts the video
renderer; leave it `"1"` if you're going to run the MuseTalk install cells below, or set it to `"0"`
to skip video and start only the voice pipeline.

In [ ]:
# 3. Session secrets and toggles
import os

os.environ["GROQ_API_KEY"] = "your_groq_key_here"  # https://console.groq.com/keys
os.environ["ENABLE_MUSETALK"] = "1"  # "0" to skip GPU lip-sync and start voice-only

# Optional: a non-Bengali voice call, or a smaller/larger Whisper checkpoint.
# os.environ["WHISPER_MODEL_SIZE"] = "large-v3"  # see backend/README.md before changing this

## 4. Mount Google Drive (optional, recommended for video calls)
Colab wipes `/content` on every session reset. Without this, the MuseTalk install below — a Python
3.10 environment plus ~7GB of model weights — downloads and rebuilds from scratch **every time your
runtime disconnects**, which on the free tier is often. Mounting Drive caches it under your Drive so
a later session's install cell finds everything already there and finishes in seconds.

Skip this cell if you're voice-only, or don't mind re-installing MuseTalk each session.

In [ ]:
# 4. Mount Drive and point the MuseTalk cache at it
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["MUSETALK_CACHE_ROOT"] = "/content/drive/MyDrive/amar_doctor_musetalk"
print("MuseTalk will install/cache under:", os.environ["MUSETALK_CACHE_ROOT"])

## 5. Doctor avatar asset (needed for video calls)
MuseTalk repaints a **photographic** mouth region — it cannot work from an illustration, and needs
a real portrait or short clip. See [`backend/static/AVATAR.md`](../backend/static/AVATAR.md) for the
framing requirements and the licence note it asks you to keep.

Prefer a **5–10s clip of the person sitting still** (`doctor_idle_source.mp4`) over a single still
image — MuseTalk cycles through prepared frames, and a still gives a frozen head with a moving jaw.

This cell reuses a copy already cached in Drive (cell 4) if one exists, so you only upload once across
sessions. Skip this cell entirely if `ENABLE_MUSETALK = "0"`.

In [ ]:
# 5. Get a doctor portrait/clip into backend/static/
import os, shutil
from pathlib import Path

STATIC_DIR = Path("backend/static")
STATIC_DIR.mkdir(parents=True, exist_ok=True)
cache_root = os.environ.get("MUSETALK_CACHE_ROOT")
asset_cache = Path(cache_root) / "avatar_assets" if cache_root else None

wanted = ["doctor_idle_source.mp4", "doctor_avatar.png"]
have = [n for n in wanted if (STATIC_DIR / n).exists()]

if have:
    print(f"Already present: {have}")
elif asset_cache and any((asset_cache / n).exists() for n in wanted):
    for n in wanted:
        src = asset_cache / n
        if src.exists():
            shutil.copyfile(src, STATIC_DIR / n)
            print(f"Restored from Drive cache: {n}")
else:
    print("Upload doctor_idle_source.mp4 (preferred) or doctor_avatar.png:")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = STATIC_DIR / name
        dest.write_bytes(data)
        print(f"Saved {dest}")
        if asset_cache:
            asset_cache.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(dest, asset_cache / name)
            print(f"  cached to Drive: {asset_cache / name}")

## 6. Install MuseTalk (one-time, ~5-10 min on a fresh runtime, seconds if cached in Drive)
Isolated on purpose: MuseTalk needs `mmcv`/`mmpose`/`mmdet` and pins `numpy==1.23.5`, which would break
the `faster-whisper` speech recognition running in Colab's own Python. So this installs into its own
Python 3.10 virtualenv, same as the local Windows recipe in `MUSETALK_SETUP.md` — Colab's own Python
(whatever the kernel is) is never touched.

Safe to re-run: every step below checks whether it already did its work and skips if so, so a dropped
connection just needs this cell run again rather than starting over.

**If `mmcv` fails to find a wheel:** it means Colab's current Python/CUDA combo has drifted from what
mmcv 2.0.1 publishes prebuilt wheels for. The error will name the missing wheel explicitly — that's a
real, occasional risk of pinning to a research repo's exact dependency versions, not a bug in this
script. `MUSETALK_SETUP.md`'s "If mmcv will not build" section describes the fallback.

In [ ]:
%%bash
set -e

CACHE_ROOT="${MUSETALK_CACHE_ROOT:-/content/amar_doctor_musetalk}"
MUSETALK_ROOT="$CACHE_ROOT/MuseTalk"
MUSETALK_VENV="$CACHE_ROOT/musetalk-venv"
echo "Cache root   : $CACHE_ROOT"
echo "MuseTalk repo: $MUSETALK_ROOT"
echo "MuseTalk venv: $MUSETALK_VENV"
mkdir -p "$CACHE_ROOT"

# ffmpeg -- Colab images ship it via apt already; this is just a safety net.
command -v ffmpeg >/dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)
echo "ffmpeg: $(command -v ffmpeg)"

# 1. Isolated Python 3.10 via uv -- Colab's own kernel Python is never touched.
pip install -q uv
if [ ! -x "$MUSETALK_VENV/bin/python" ]; then
  echo "--- Creating Python 3.10 venv ---"
  uv python install 3.10
  uv venv --python 3.10 "$MUSETALK_VENV"
else
  echo "--- venv already exists, skipping ---"
fi
PY="$MUSETALK_VENV/bin/python"

# 2. Clone MuseTalk
if [ ! -d "$MUSETALK_ROOT/.git" ]; then
  echo "--- Cloning MuseTalk ---"
  git clone --depth 1 https://github.com/TMElyralab/MuseTalk "$MUSETALK_ROOT"
else
  echo "--- MuseTalk repo already cloned, skipping ---"
fi

# 3. torch + the mmcv/mmdet/mmpose stack, in dependency order (mim resolves
#    mmcv against whichever torch is already installed).
if ! "$PY" -c "import mmpose" >/dev/null 2>&1; then
  echo "--- Installing torch 2.0.1+cu118 ---"
  "$PY" -m pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
  "$PY" -m pip install -q -U openmim
  # mim (unmaintained) needs pkg_resources, which recent setuptools dropped;
  # chumpy's setup.py needs `wheel` present to build at all. Both bit the
  # local Windows install too -- see MUSETALK_SETUP.md.
  "$PY" -m pip install -q "setuptools<70" wheel
  echo "--- mim install mmengine ---"
  "$PY" -m mim install mmengine
  echo "--- mim install mmcv==2.0.1 ---"
  "$PY" -m mim install "mmcv==2.0.1"
  echo "--- chumpy (mmpose dependency with a broken build-isolation setup.py) ---"
  "$PY" -m pip install -q --no-build-isolation chumpy
  echo "--- mim install mmdet==3.1.0 ---"
  "$PY" -m mim install "mmdet==3.1.0"
  echo "--- mim install mmpose==1.1.0 ---"
  "$PY" -m mim install "mmpose==1.1.0"
else
  echo "--- mmcv/mmdet/mmpose already installed, skipping ---"
fi

# 4. MuseTalk's own requirements, minus training-only/UI extras we don't run headless.
if ! "$PY" -c "import diffusers" >/dev/null 2>&1; then
  echo "--- Installing MuseTalk's own requirements (headless) ---"
  grep -vE '^(tensorflow|tensorboard|gradio)' "$MUSETALK_ROOT/requirements.txt" > "$MUSETALK_ROOT/requirements-headless.txt"
  "$PY" -m pip install -q -r "$MUSETALK_ROOT/requirements-headless.txt"
  "$PY" -m pip install -q "huggingface_hub[cli]==0.30.2" fastapi "uvicorn[standard]" httpx
else
  echo "--- MuseTalk requirements already installed, skipping ---"
fi

# 5. Weights (~7.3GB total). Default HF endpoint -- MuseTalk's own
#    download_weights.sh points at a China mirror; deliberately not used here.
CLI="$MUSETALK_VENV/bin/huggingface-cli"
if [ ! -f "$MUSETALK_ROOT/models/musetalkV15/unet.pth" ]; then
  echo "--- Downloading weights (~7.3GB) ---"
  "$CLI" download TMElyralab/MuseTalk       --local-dir "$MUSETALK_ROOT/models"
  "$CLI" download yzd-v/DWPose              --local-dir "$MUSETALK_ROOT/models/dwpose"  --include "dw-ll_ucoco_384.pth"
  "$CLI" download stabilityai/sd-vae-ft-mse --local-dir "$MUSETALK_ROOT/models/sd-vae"  --include "config.json" "diffusion_pytorch_model.bin"
  "$CLI" download openai/whisper-tiny       --local-dir "$MUSETALK_ROOT/models/whisper" --include "config.json" "pytorch_model.bin" "preprocessor_config.json"
  "$CLI" download ManyOtherFunctions/face-parse-bisent --local-dir "$MUSETALK_ROOT/models/face-parse-bisent" --include "79999_iter.pth" "resnet18-5c106cde.pth"
else
  echo "--- Weights already downloaded, skipping ---"
fi

echo ""
echo "MuseTalk environment ready at $MUSETALK_ROOT"

## 7. Launch backend + (if installed) MuseTalk + tunnel
`colab_runner.py` installs the main backend's own dependencies, starts the MuseTalk renderer in the
background **only if** cells 5 and 6 actually completed (otherwise it says so and continues without
video), starts the FastAPI backend, and opens the Cloudflare tunnel. Watch the output for:

- `✓ MuseTalk renderer ready` — video calls get real GPU lip-sync
- `ℹ️ ... skipping GPU lip-sync` — video calls use the audio-reactive avatar; it names which of cells 5/6 to check

Then copy the `https://....trycloudflare.com` URL from the output and paste it into the "Connect
Backend" box on the `/chat` page of your locally-running Next.js app.

In [ ]:
# 7. Launch backend & Cloudflare tunnel
!python3 backend/colab_runner.py